In [4]:
import pandas as pd
from pathlib import Path

In [5]:
df = pd.read_csv("../data/raw/lebron_game_logs.csv")

In [6]:
df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
df = df.sort_values("GAME_DATE").reset_index(drop=True)

In [7]:
stats = ["PTS", "REB", "AST", "MIN"]

for stat in stats:
    # previous game
    df[f"{stat}_LAG_1"] = df[stat].shift(1)
    
    # avgs of the 5 and 10 previous games
    df[f"{stat}_AVG_5"] = (
        df[stat].shift(1).rolling(window=5).mean()
    )
    
    df[f"{stat}_AVG_10"] = (
        df[stat].shift(1).rolling(window=10).mean()
    )

# rest days
df["REST_DAYS"] = (
    df.groupby("SEASON_ID")["GAME_DATE"]
    .diff()
    .dt.days - 1
)

# home vs away
df["HOME"] = (
    df["MATCHUP"].str.contains("vs.", regex=False)
).astype(int)

df.head(15)

,SEASON_ID,Player_ID,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,...,REB_AVG_5,REB_AVG_10,AST_LAG_1,AST_AVG_5,AST_AVG_10,MIN_LAG_1,MIN_AVG_5,MIN_AVG_10,REST_DAYS,HOME
0,22015,2544,21500002,2015-10-27,CLE @ CHI,L,36,12,22,0.545,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,22015,2544,21500011,2015-10-28,CLE @ MEM,W,31,4,13,0.308,...,NaN,NaN,5.0,NaN,NaN,36.0,NaN,NaN,0.0,0
2,22015,2544,21500021,2015-10-30,CLE vs. MIA,W,34,13,19,0.684,...,NaN,NaN,5.0,NaN,NaN,31.0,NaN,NaN,1.0,1
3,22015,2544,21500046,2015-11-02,CLE @ PHI,W,33,9,19,0.474,...,NaN,NaN,4.0,NaN,NaN,34.0,NaN,NaN,2.0,0
4,22015,2544,21500063,2015-11-04,CLE vs. NYK,W,35,9,23,0.391,...,NaN,NaN,11.0,NaN,NaN,33.0,NaN,NaN,1.0,1
5,22015,2544,21500078,2015-11-06,CLE vs. PHI,W,36,12,22,0.545,...,7.2,NaN,3.0,5.6,NaN,35.0,33.8,NaN,1.0,1
6,22015,2544,21500094,2015-11-08,CLE vs. IND,W,35,10,23,0.435,...,6.0,NaN,13.0,7.2,NaN,36.0,33.8,NaN,1.0,1
7,22015,2544,21500106,2015-11-10,CLE vs. UTA,W,38,11,19,0.579,...,5.8,NaN,4.0,7.0,NaN,35.0,34.6,NaN,1.0,1
8,22015,2544,21500130,2015-11-13,CLE @ NYK,W,39,12,21,0.571,...,6.2,NaN,8.0,7.8,NaN,38.0,35.4,NaN,2.0,0
9,22015,2544,21500141,2015-11-14,CLE @ MIL,L,45,13,27,0.481,...,5.0,NaN,6.0,6.8,NaN,39.0,36.6,NaN,0.0,0


In [8]:
# FEATURES
feature_cols = [
    "PTS_LAG_1", "PTS_AVG_5", "PTS_AVG_10",
    "REB_LAG_1", "REB_AVG_5", "REB_AVG_10",
    "AST_LAG_1", "AST_AVG_5", "AST_AVG_10",
    "MIN_LAG_1", "MIN_AVG_5", "MIN_AVG_10",
    "REST_DAYS", "HOME"
]

# TARGETS
target_cols = ["PTS", "REB", "AST", "MIN"]

# REMOVING LINES WITH MISSING DATA
df_model = df.dropna(
    subset=feature_cols + target_cols
).copy()

# CONVERTS TO INT
df_model["REST_DAYS"] = df_model["REST_DAYS"].astype(int)

print(f"Original dataframe: {len(df)} lines")
print(f"Dataframe post cleaning: {len(df_model)} lines")
print(f"Remaining nulls: {df_model[feature_cols + target_cols].isna().sum().sum()}")

Original dataframe: 711 lines
Dataframe post cleaning: 691 lines
Remaining nulls: 0


In [9]:
project_root = Path.cwd().resolve()
output_dir = project_root / "data" / "raw"

if not output_dir.exists():
    output_dir = project_root.parent / "data" / "raw"

output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "lebron_game_logs_model.csv"

df_model.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)